In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import StratifiedKFold

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
TABLES_DIR = RESULTS_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
survival = pd.read_csv(DATA_DIR / "survival_luminal_clean.csv")

print(survival.shape)
survival.head()

(556, 7)


,patient,BRCA_Subtype_PAM50,vital_status,days_to_death,days_to_last_follow_up,event,time
0,TCGA-3C-AAAU,LumA,Alive,NaN,4047.0,0,4047.0
1,TCGA-3C-AALJ,LumB,Alive,NaN,1474.0,0,1474.0
2,TCGA-3C-AALK,LumA,Alive,NaN,1448.0,0,1448.0
3,TCGA-4H-AAAK,LumA,Alive,NaN,348.0,0,348.0
4,TCGA-5L-AAT0,LumA,Alive,NaN,1477.0,0,1477.0


In [3]:
print(survival["BRCA_Subtype_PAM50"].value_counts())
print(survival["event"].value_counts())

BRCA_Subtype_PAM50
LumA    416
LumB    140
Name: count, dtype: int64
event
0    491
1     65
Name: count, dtype: int64


In [4]:
survival["stratum"] = (
    survival["BRCA_Subtype_PAM50"].astype(str)
    + "_event_"
    + survival["event"].astype(str)
)

survival["stratum"].value_counts()

stratum
LumA_event_0    376
LumB_event_0    115
LumA_event_1     40
LumB_event_1     25
Name: count, dtype: int64

In [5]:
stratum_counts = survival["stratum"].value_counts()

print(stratum_counts)

if (stratum_counts < 5).any():
    print("Warning: at least one stratum has fewer than 5 patients.")
else:
    print("All strata have enough patients for 5-fold CV.")

stratum
LumA_event_0    376
LumB_event_0    115
LumA_event_1     40
LumB_event_1     25
Name: count, dtype: int64
All strata have enough patients for 5-fold CV.


In [6]:
N_SPLITS = 5
RANDOM_STATE = 42

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

survival["fold"] = -1

for fold_number, (_, validation_idx) in enumerate(
    skf.split(survival, survival["stratum"]),
    start=1
):
    survival.loc[validation_idx, "fold"] = fold_number

survival["fold"].value_counts().sort_index()

fold
1    112
2    111
3    111
4    111
5    111
Name: count, dtype: int64

In [7]:
assert (survival["fold"] != -1).all()
assert survival["patient"].is_unique

print("Every patient has exactly one fold assignment.")

Every patient has exactly one fold assignment.


In [8]:
fold_distribution = (
    survival
    .groupby(["fold", "BRCA_Subtype_PAM50", "event"])
    .size()
    .reset_index(name="n_patients")
    .sort_values(["fold", "BRCA_Subtype_PAM50", "event"])
)

fold_distribution

,fold,BRCA_Subtype_PAM50,event,n_patients
0,1,LumA,0,76
1,1,LumA,1,8
2,1,LumB,0,23
3,1,LumB,1,5
4,2,LumA,0,75
5,2,LumA,1,8
6,2,LumB,0,23
7,2,LumB,1,5
8,3,LumA,0,75
9,3,LumA,1,8


In [10]:
fold_counts = (
    survival
    .groupby(["fold", "BRCA_Subtype_PAM50", "event"])
    .size()
    .reset_index(name="n_patients")
)

fold_totals = (
    survival
    .groupby("fold")
    .size()
    .reset_index(name="fold_total")
)

fold_proportions = fold_counts.merge(fold_totals, on="fold")
fold_proportions["proportion"] = (
    fold_proportions["n_patients"] / fold_proportions["fold_total"]
)

fold_proportions = fold_proportions.sort_values(
    ["fold", "BRCA_Subtype_PAM50", "event"]
)

fold_proportions

,fold,BRCA_Subtype_PAM50,event,n_patients,fold_total,proportion
0,1,LumA,0,76,112,0.678571
1,1,LumA,1,8,112,0.071429
2,1,LumB,0,23,112,0.205357
3,1,LumB,1,5,112,0.044643
4,2,LumA,0,75,111,0.675676
5,2,LumA,1,8,111,0.072072
6,2,LumB,0,23,111,0.207207
7,2,LumB,1,5,111,0.045045
8,3,LumA,0,75,111,0.675676
9,3,LumA,1,8,111,0.072072


In [12]:
cv_fold_assignments = survival[[
    "patient",
    "BRCA_Subtype_PAM50",
    "event",
    "time",
    "fold"
]].copy()

cv_fold_assignments.head()

,patient,BRCA_Subtype_PAM50,event,time,fold
0,TCGA-3C-AAAU,LumA,0,4047.0,4
1,TCGA-3C-AALJ,LumB,0,1474.0,5
2,TCGA-3C-AALK,LumA,0,1448.0,5
3,TCGA-4H-AAAK,LumA,0,348.0,4
4,TCGA-5L-AAT0,LumA,0,1477.0,3


In [13]:
cv_fold_assignments.to_csv(
    DATA_DIR / "cv_fold_assignments.csv",
    index=False
)

fold_distribution.to_csv(
    TABLES_DIR / "cv_fold_distribution.csv",
    index=False
)

fold_proportions.to_csv(
    TABLES_DIR / "cv_fold_proportions.csv",
    index=False
)